<a href="https://colab.research.google.com/github/anadelrio18/emociones-voz/blob/main/Base_de_datos_italiano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import librosa
from skimage.filters import threshold_otsu

# Descargar dataset italiano
ruta_base = kagglehub.dataset_download("anasofiadelrio/audios-exp3-italiano")
CARPETA_italiano = Path(ruta_base)

# Verificar
wavs = list(CARPETA_italiano.glob("**/*.wav"))
print(f"Audios encontrados: {len(wavs)}")
print(f"Ejemplo: {wavs[0].name}")

100%|██████████| 239M/239M [00:01<00:00, 153MB/s]

Extracting files...


Audios encontrados: 588
Ejemplo: sor-m3-n4.wav


In [ ]:
EMOCIONES_ITALIANO = {
    'dis': 'Disgusto',
    'pau': 'Miedo',
    'rab': 'Ira',
    'gio': 'Felicidad',
    'sor': 'Sorpresa',
    'tri': 'Tristeza',
    'neu': 'Neutral'
}

In [ ]:
#1) suprimimos el pixel central del espectro 2D que corresponde a la frecuencia espacial 0, ese pixel representa el valor promedio de toda la imagen (cuanta energía total tiene el espectrograma)

#2) Normalizamos con z-score que transforma el perfil para que tenga media 0 y desviación estandar 1
# La formula es simplemente restar la media y dividir por la desviación estándar
#---> así el detector compara patrones, no volumenes
def superprocesador(carpeta, codigo):

    archivos = [
        f for f in carpeta.glob("**/*.wav")
        if f.stem.split("-")[0].lower()==codigo
    ]

    print(f"{codigo}: {len(archivos)} audios")

    imagenes=[]

    for archivo in archivos:

        audio,sr=librosa.load(archivo,sr=None)

        n=int(sr*2.5)

        if len(audio)>=n:
            audio=audio[:n]
        else:
            audio=np.pad(audio,(0,n-len(audio)))

        stft=librosa.stft(audio)

        espectrograma=np.abs(stft)

        espectrograma_db=librosa.amplitude_to_db(
            espectrograma,
            ref=np.max
        )

        umbral=threshold_otsu(espectrograma_db)

        binaria=espectrograma_db>umbral

        fft2=np.fft.fftshift(
            np.fft.fft2(binaria)
        )

        modulo=np.log1p(
            np.abs(fft2)
        )

        imagenes.append(modulo)

    promedio=np.mean(imagenes,axis=0)

    cy=promedio.shape[0]//2
    cx=promedio.shape[1]//2

    promedio[cy,cx]=0

    perfil=promedio[:,cx]

    perfil=(perfil-perfil.mean())/perfil.std()

    return perfil

In [ ]:
firmas={}

for codigo in EMOCIONES_ITALIANO:

    perfil=superprocesador(
        CARPETA_italiano,
        codigo
    )

    firmas[codigo]={

        "perfil":perfil,

        "nombre":EMOCIONES_ITALIANO[codigo]

    }

dis: 84 audios
pau: 84 audios
rab: 84 audios
gio: 84 audios
sor: 84 audios
tri: 84 audios
neu: 84 audios


In [ ]:
#Correlación de Pearson que mide que tan parecida es la forma de dos curvas, comparar la forma de los perfiles
def detectar_emocion(ruta_audio):

    audio,sr=librosa.load(ruta_audio,sr=None)

    n=int(sr*2.5)

    if len(audio)>=n:
        audio=audio[:n]
    else:
        audio=np.pad(audio,(0,n-len(audio)))

    stft=librosa.stft(audio)

    espectrograma=np.abs(stft)

    espectrograma_db=librosa.amplitude_to_db(
        espectrograma,
        ref=np.max
    )

    umbral=threshold_otsu(espectrograma_db)

    binaria=espectrograma_db>umbral

    fft2=np.fft.fftshift(
        np.fft.fft2(binaria)
    )

    modulo=np.log1p(
        np.abs(fft2)
    )

    cy=modulo.shape[0]//2
    cx=modulo.shape[1]//2

    modulo[cy,cx]=0

    perfil=modulo[:,cx]

    perfil=(perfil-perfil.mean())/perfil.std()

    correlaciones={}

    for codigo,datos in firmas.items():

        correlacion=np.corrcoef(
            perfil,
            datos["perfil"]
        )[0,1]

        correlaciones[codigo]=correlacion

    ganador=max(
        correlaciones,
        key=correlaciones.get
    )

    return firmas[ganador]["nombre"]

In [ ]:
correctos=0
total=0

for codigo,datos in firmas.items():

    archivos=list(
        CARPETA_italiano.glob(f"{codigo}-*.wav")
    )

    ok=0

    for archivo in archivos:

        pred=detectar_emocion(archivo)

        if pred==datos["nombre"]:

            ok+=1
            correctos+=1

        total+=1

    print(datos["nombre"],ok,"/",len(archivos))

print(
    "Precisión:",
    100*correctos/total,
    "%"
)

Disgusto 72 / 84
Miedo 71 / 84
Ira 56 / 84
Felicidad 72 / 84
Sorpresa 73 / 84
Tristeza 66 / 84
Neutral 69 / 84
Precisión: 81.4625850340136 %


In [ ]:
from google.colab import files
from IPython.display import Audio, display

def subir_y_detectar():
    print("📁 Selecciona un archivo de audio .wav")
    uploaded = files.upload()

    for nombre, contenido in uploaded.items():
        ruta = Path(f"/content/{nombre}")
        ruta.write_bytes(contenido)
        print(f"✅ Archivo recibido: {nombre}")
        display(Audio(str(ruta)))

        print("\n🔍 Detectando emoción...")
        emocion = detectar_emocion(ruta)
        print(f"\n{'='*45}")
        print(f"  Emoción detectada: *** {emocion.upper()} ***")
        print(f"{'='*45}")

subir_y_detectar()

📁 Selecciona un archivo de audio .wav


Saving audioclip-1783642569000-3627.ogg to audioclip-1783642569000-3627.ogg
✅ Archivo recibido: audioclip-1783642569000-3627.ogg



🔍 Detectando emoción...

  Emoción detectada: *** MIEDO ***
